# TabPFN-3.5 On-Premises Classifier - DSDL Notebook Template

This notebook implements an on-premises TabPFN-3.5 classifier on top of the Splunk App for Data Science and Deep Learning (DSDL) - formerly known as the Deep Learning Toolkit for Splunk (DLTK). It is based on DSDL's barebone example template; find more background and other algorithm examples in the app itself and on the [DSDL Splunkbase page](https://splunkbase.splunk.com/app/4607/#/details).

**Important:** Every time you save this notebook, only the specifically tagged cells below (`mltkc_import`, `mltkc_init`, `mltkc_fit`, `mltkc_apply`, `mltkc_save`, `mltkc_load`, `mltkc_summary`) are exported into the Python module that actually serves `| fit` / `| apply` / `| summary` calls from Splunk. Any import, constant, or helper function defined in a free/untagged cell - including this notebook's own testing cells above and below - is invisible to production `fit`/`apply` calls, even though it runs fine when you execute it interactively here. **Put anything your model logic depends on into `mltkc_import`, not into a free cell above or below it.** Never use the cell type *Raw* for notes or commands: raw cells can end up in the exported module and break it - use Markdown code fences instead. See the Model Development Guide in the DSDL app for more on this workflow.

**License:** the TabPFN-3.5 weights are released under a non-commercial license. Testing and internal evaluation are allowed, commercial or production use is not - read the license (the `LICENSE` file in the model repository) before going beyond evaluation.

## Stage -1 - one-time setup: download the TabPFN-3.5 checkpoint (run once, not exported)

TabPFN's PyPI package ships no model weights - the transformer checkpoint (one file, about 0.9 GB, for classification and regression) lives in Prior Labs' Hugging Face repository `Prior-Labs/tabpfn_3_5`. We download it explicitly, once, and load it from a local file afterwards via `model_path=` - so that `fit()`/`apply()` make **zero network calls** at runtime. That's what makes this deployment genuinely on-premises rather than just cloud-API-optional.

Run the cell below once. At the time of writing the download works without a token; if your network rejects it or Hugging Face asks for authentication, create a *read* token (https://huggingface.co/settings/tokens/new?preset=read-only) and paste it into `HF_TOKEN`.

If this fails with `SSLCertVerificationError`, the container's network path to huggingface.co is most likely intercepted by a proxy it doesn't trust. In that case, skip this cell, download `tabpfn-v3.5-20260909.safetensors` once via a normal browser on the host machine instead, and copy it into the container manually, e.g. `docker cp <file> <container_id>:/srv/app/model/data/`.

In [ ]:
# THIS CELL IS NOT EXPORTED - one-time setup, run manually when needed

HF_TOKEN = None  # optional: only needed if the download is rejected or rate-limited
HF_REPO_ID = "Prior-Labs/tabpfn_3_5"
HF_FILENAME = "tabpfn-v3.5-20260909.safetensors"  # keep in sync with MODEL_PATH in the Stage 0 cell below
HF_LOCAL_DIR = "/srv/app/model/data/"

from huggingface_hub import hf_hub_download

downloaded_path = hf_hub_download(
    repo_id=HF_REPO_ID,
    filename=HF_FILENAME,
    token=HF_TOKEN,
    local_dir=HF_LOCAL_DIR,
)
print(f"Checkpoint ready at: {downloaded_path}")

## Stage 0 - import libraries
At stage 0 we define all imports necessary to run our subsequent code, including `tabpfn` itself and the local checkpoint path used throughout this notebook (the file downloaded in Stage -1).

In [ ]:
# this definition exposes all python module imports that should be available in all subsequent commands
import json
import numpy as np
import pandas as pd
import os
import pickle

import tabpfn
from tabpfn import TabPFNClassifier, TabPFNRegressor

# global constants
MODEL_DIRECTORY = "/srv/app/model/data/"
MODEL_PATH = MODEL_DIRECTORY + "tabpfn-v3.5-20260909.safetensors"  # keep in sync with HF_FILENAME in Stage -1

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
print("numpy version: " + np.__version__)
print("pandas version: " + pd.__version__)
print("tabpfn version: " + tabpfn.__version__)
print("checkpoint found: " + str(os.path.exists(MODEL_PATH)))

## Stage 1 - get a data sample from Splunk
There are currently 2 ways to retrieve data from Splunk: Option 1 is to interactively pull data from Splunk into the DSDL Jupyter Lab environment. This is useful when the Splunk REST API is accessible from the Jupyter environment and a valid Splunk auth token is defined in the DSDL app. This option has advantages to quickly experiment with different Splunk SPL queries and further interactively work with the search results in Jupyter. Note that no meta data param object is retrieved here. This is only available in Stage 1 Option 2.

### Option 1 - pull data from Splunk
(Not used in this notebook's own development - shown here for completeness. We used Option 2 below throughout, since it also gives us the metadata `param` object that `fit()`/`apply()` need.)

In [ ]:
from dsdlsupport import SplunkSearch as SplunkSearch

In [ ]:
search = SplunkSearch.SplunkSearch()

In [ ]:
df = search.as_df()
df

### Option 2 - push data from Splunk
In Splunk run a search to pipe a dataset into your notebook environment. You utilize the `mode=stage` flag in the `| fit` command to do this. The search results are accessible then as a csv file with the same model name that is defined in the `into app:<modelname>` part of the fit statement. Additionally, meta data is retrieved and accessible as a json file or the `param` object. In the same way you can further work with the meta data object as it is exposed in the fit and apply function definitions below in stage 3 and 4.

| makeresults count=50<br>
| eval f1=random()%100, f2=random()%100, target=if(f1>50,1,0)<br>
| fit MLTKContainer mode=stage algo=tabpfn_model target from f1 f2 into app:tabpfn_model<br>

After you run this search your data set sample is available as a csv (and the accompanying metadata as json) inside the container to develop your model. Both file names are taken from the `into app:<modelname>` keyword ("tabpfn_model" in the example above), or set to "default" if no `into` keyword is present. This step is intended to work with a subset of your data to create your custom model.

In [ ]:
# this cell is not executed from MLTK and should only be used for staging data into the notebook environment
def stage(name):
    with open("data/"+name+".csv", 'r') as f:
        df = pd.read_csv(f)
    with open("data/"+name+".json", 'r') as f:
        param = json.load(f)
    return df, param

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
df, param = stage("tabpfn_model")

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
df

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
param

## Stage 2 - create and initialize a model
For this integration, `init()` doesn't need to set anything up - TabPFN is a frozen, pre-trained model with no per-model state to initialize before `fit()` is called.

In [ ]:
def init(df, param):
    model = {}
    return model

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
model = init(df,param)
print(model)

## Stage 3 - fit the model

Note: TabPFN's transformer weights are frozen and pre-trained - there is no gradient descent here. "Fit" caches the labeled training rows so the model can use them as in-context examples during `apply()` later. That's why `fit()` below only stores the training data and the target column name, not a trained classifier object.

In [ ]:
def fit(model, df, param):
    X = df[param['feature_variables']]
    y = df[param['target_variables'][0]]
    model['X_train'] = X
    model['y_train'] = y
    model['target_name'] = param['target_variables'][0]
    return model

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
print(fit(model,df,param))

## Stage 4 - apply the model

`apply()` reconstructs a fresh `TabPFNClassifier` from the on-disk checkpoint (`MODEL_PATH`, from Stage 0) and re-supplies the cached training rows on every call, rather than reusing a classifier object saved from `fit()`. This keeps what Splunk persists in its own model storage (see Stage 5) small - the full loaded classifier would carry the entire checkpoint in memory and risk exceeding Splunk's default 30 MB per-model size limit (`mlspl_limits.max_model_size_mb`).

In [ ]:
def apply(model, df, param):
    X = df[param['feature_variables']]
    clf = TabPFNClassifier(model_path=MODEL_PATH)
    #clf = TabPFNClassifier(model_path=MODEL_PATH, ignore_pretraining_limits=True)
    clf.fit(model['X_train'], model['y_train'])
    preds = clf.predict(X)
    return pd.DataFrame({'predicted_' + model['target_name']: preds})

In [ ]:
# THIS CELL IS NOT EXPORTED - free notebook cell for testing or development purposes
print(apply(model,df,param))

## Stage 5 - save the model

Only `X_train`, `y_train`, and the target column name are pickled here - deliberately not the TabPFN weights themselves, which stay on disk as the checkpoint file and are loaded fresh by `apply()` each time.

In [ ]:
def save(model, name):
    with open(f"{MODEL_DIRECTORY}{name}.pkl", "wb") as f:
        pickle.dump(model, f)
    return model

## Stage 6 - load the model
Reconstructs the small pickled dict (`X_train`, `y_train`, `target_name`) saved in Stage 5 - not the TabPFN weights, which are loaded separately by `apply()` from `MODEL_PATH`.

In [ ]:
def load(name):
    with open(f"{MODEL_DIRECTORY}{name}.pkl", "rb") as f:
        return pickle.load(f)

## Stage 7 - provide a summary of the model

In [ ]:
def summary(model=None):
    return {"tabpfn_version": tabpfn.__version__}

**Example: predict malware-associated traffic from session metrics**

`firewall_traffic.csv` is a Splunk-collected AI Toolkit sample dataset (free to use, no citation required). `used_by_malware` (yes/no) is nicely balanced (~59% yes / 41% no across ~99k sessions) - checked via `| inputlookup firewall_traffic.csv | stats count by used_by_malware` before committing to it, since `hostperf.csv`'s `pl` field turned out to be a 0.3%-positive outlier-detection field, not a usable classification target. Predicted from `bytes_received`/`bytes_sent`/`packets_received`/`packets_sent`. ~99k rows is far more than TabPFN accepts on a CPU by default, so we draw a reproducible sample (fixed seeds) and split it into a training half and a held-out half:<br>

| inputlookup firewall_traffic.csv<br>
| eval malware_label=if(used_by_malware=="yes",1,0)<br>
| sample seed=123 count=2000<br>
| sample partitions=2 fieldname=group seed=1234<br>
| where group=0<br>
| fit MLTKContainer algo=tabpfn_model malware_label from bytes_received bytes_sent packets_received packets_sent into app:tabpfn_firewall<br>

Then apply the model to the held-out half and score it:

| inputlookup firewall_traffic.csv<br>
| eval malware_label=if(used_by_malware=="yes",1,0)<br>
| sample seed=123 count=2000<br>
| sample partitions=2 fieldname=group seed=1234<br>
| where group=1<br>
| apply tabpfn_firewall<br>
| `confusionmatrix(malware_label, predicted_malware_label)`

## End of Stages
All subsequent cells are not tagged and can be used for further freeform code